In [ ]:
# === Mount Google Drive ===
from google.colab import drive
drive.mount('/content/drive')

# === Install necessary packages (only run once) ===
!pip install -q mne plotly colorama pyphen

# === Imports ===
import os
import re
import gc
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import pyphen
import mne
import plotly.graph_objects as go
import xgboost as xgb


from IPython.display import display
from colorama import Fore, Style, init

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from xgboost import XGBClassifier

# === Colorama init (for colored terminal printouts) ===
init(autoreset=True)



# Predicting Perception from Acoustic & ERP Signals

**Research question.** Can acoustic (speech rate, MFCCs, F0, RMS, ZCR, spectral centroid) and ERP (N1, N280) features predict correct perception above chance?



# Hypothesis:
Null Hypothesis ($H_0$):
ERP and acoustic features (e.g., speech rate, inter-word interval, MFCCs, F0, etc.) do not predict participants’ perception accuracy better than chance.


Alternative Hypothesis ($H_1$):
ERP and acoustic features do predict perception accuracy significantly better than chance.



# Table of Contents
- Hypotheses
- Data Preprocessing
- Feature Engineering
- Modeling & Classification
- Validation
- Results & Interpretation



# 1. Data Preprocessing

## Behavioral Data Preprocessing




In [ ]:
# === 1. Parse Sentences_List1.txt ===
def parse_sentences_from_txt(txt_file):
    with open(txt_file, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    data = []
    for line in lines:
        line = line.strip()
        if line and line[0].isdigit() and '|' in line:
            try:
                idx, rest = line.split('.', 1)
                words = rest.strip().split()

                word1_raw = None
                function_word = None
                word2 = None

                for i, word in enumerate(words):
                    if '|' in word:
                        word1_raw = word
                        if i + 1 < len(words):
                            function_word = words[i + 1]
                        if i + 2 < len(words):
                            word2 = words[i + 2].replace('|', '')
                        break

                if word1_raw and function_word:
                    word1_clean = word1_raw.replace('|', '').lower()
                    function_word = function_word.lower()
                    full_sentence = ' '.join(w.replace('|', '') for w in words)

                    data.append({
                        'Index': int(idx),
                        'Word1': word1_clean,
                        'FunctionWord': function_word,
                        'Word2': word2.lower() if word2 else None,
                        'Full_Sentence': full_sentence.lower()
                    })

            except Exception as e:
                print(f"⚠️ Failed to parse line: {line}\n{e}")
                continue

    return pd.DataFrame(data)

# === 2. Check response logic ===
def check_sequence_after_word1(row, sentence_df):
    stim_text = str(row.get("StimTest.RESP", "")).lower().strip()
    stim_words = stim_text.split()

    idx = row.get("Index")
    match = sentence_df[sentence_df["Index"] == idx]
    if match.empty:
        return None

    word1 = match.iloc[0]["Word1"]
    func_word = match.iloc[0]["FunctionWord"]
    word2 = match.iloc[0]["Word2"]

    try:
        i = stim_words.index(word1)
    except ValueError:
        return None  # word1 not found

    if i + 1 < len(stim_words) and stim_words[i + 1] == func_word:
        return 0
    if i + 1 < len(stim_words) and stim_words[i + 1] == word2:
        return 1
    return None

# === 3. Smart CSV reader ===
def smart_read_csv(filepath):
    for enc in ['utf-8', 'utf-16', 'windows-1252']:
        for delim in [',', '\t', ';']:
            try:
                df = pd.read_csv(filepath, delimiter=delim, encoding=enc, engine='python', on_bad_lines='skip')
                if df.shape[1] >= 2:
                    return df
            except Exception:
                continue
    return None

# === 4. Main function with save ===
def print_evaluation_of_participants(root_dir, sentence_file):
    sentence_df = parse_sentences_from_txt(sentence_file)
    all_dfs = []  # Collect for saving

    for participant_num in range(2, 28):
        folder_path = os.path.join(root_dir, str(participant_num))
        if not os.path.isdir(folder_path):
            print(f"❌ Folder not found: {folder_path}")
            continue

        print(f"\n👤 Processing Participant {participant_num}")

        for fname in os.listdir(folder_path):
            if fname.lower().startswith("speechrate") and fname.lower().endswith(".csv"):
                input_path = os.path.join(folder_path, fname)
                df = smart_read_csv(input_path)

                if df is None:
                    print(f"❌ Could not read: {fname}")
                    continue

                df.rename(columns={"WavFiler": "Index", "CellT": "Condition"}, inplace=True)

                if not all(c in df.columns for c in ["Index", "Condition", "StimTest.RESP"]):
                    print(f"⚠️ Unexpected columns in {fname}")
                    continue

                df["Index"] = df["Index"].astype("Int64", errors="ignore")
                df = pd.merge(df, sentence_df, on="Index", how="left")

                df["Correct_Sequence (0/1)"] = df.apply(lambda row: check_sequence_after_word1(row, sentence_df), axis=1)
                df["Correct_Sequence (0/1)"] = df["Correct_Sequence (0/1)"].astype("Int64")
                df["Include_For_Stats"] = df["Correct_Sequence (0/1)"].isin([0, 1])
                df["Participant"] = participant_num

                all_dfs.append(df)
                print(df[["Index", "Condition", "StimTest.RESP", "Correct_Sequence (0/1)", "Include_For_Stats"]].head(10))
                break

    # Combine and save
    if all_dfs:
        full_df = pd.concat(all_dfs, ignore_index=True)
        save_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/All_Participants_Summary_includeNA.csv"
        full_df.to_csv(save_path, index=False)
        print(f"\n✅ Saved full dataset with NAs: {save_path}")
    else:
        print("⚠️ No valid participant data to save.")

# === 5. Run ===
if __name__ == "__main__":
    root_dir = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Behavioral"
    sentence_file = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/Sentences_List1.txt"
    print_evaluation_of_participants(root_dir, sentence_file)






| Value  | Meaning                                                         |
| ------ | --------------------------------------------------------------- |
| `0`    | **Correct** — The function word followed the expected `Word1` |
| `1`    | **Incorrect** — The function word **did not** follow `Word1`  |
| `<NA>` | **Not applicable** — One of the following:                   |
|        | - The response was too short                                    |
|        | - The sentence was malformed or incomplete                      |


## Load Onset Times



In [ ]:
csv_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/SR_OnsetTimes_Final.csv"
df = pd.read_csv(csv_path)
print(df.head())



### Check If the Onset Times are in ms or seconds

In [ ]:


# === Set paths ===
audio_dir = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli"

# === Check a sample audio file ===
sample_file = "SR129.wav"
audio_path = os.path.join(audio_dir, sample_file)

# === Load audio and print duration ===
y, sr = librosa.load(audio_path, sr=16000)  # 16kHz
duration = len(y) / sr
print(f"\n🔊 '{sample_file}' duration: {duration:.2f} seconds")

# === Check if onset values are in ms or sec ===
onset_sample_ms = df.loc[0, "critical"]  # or any other column
onset_sample_sec = onset_sample_ms / 1000
print(f"Sample 'critical' onset (converted): {onset_sample_sec:.3f} sec")

# === Conclusion ===
if onset_sample_sec < duration:
    print("✅ Onset values are in milliseconds.")
else:
    print("⚠️ Onset values might already be in seconds — please double check.")



## Merge Behavioral & Timing Data

In [ ]:

# === Load behavioral data ===
behavior_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/All_Participants_Summary_includeNA.csv"
df_behavior = pd.read_csv(behavior_path)

# === Load onset timing data ===
onset_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/SR_OnsetTimes_Final.csv"
df_onset = pd.read_csv(onset_path)

# === Clean column names ===
df_behavior.columns = df_behavior.columns.str.strip()
df_onset.columns = df_onset.columns.str.strip()

# === Rename onset columns for consistency ===
df_onset = df_onset.rename(columns={
    "sentence#": "Index",
    "Word1": "Word1_Onset",
    "critical": "FunctionWord_Onset",
    "Word2": "Word2_Onset",
    "NorE": "Condition"
})

# === Ensure 'Index' and 'Condition' are of the same type and consistent ===
df_behavior["Index"] = df_behavior["Index"].astype(int)
df_onset["Index"] = df_onset["Index"].astype(int)

df_behavior["Condition"] = df_behavior["Condition"].str.upper().str.strip()
df_onset["Condition"] = df_onset["Condition"].str.upper().str.strip()

# === Merge on Index + Condition to avoid duplication ===
df_merged = pd.merge(df_behavior, df_onset, on=["Index", "Condition"], how="left")

# ✅ Convert key columns to nullable integers
int_cols = ["Correct_Sequence (0/1)", "Word1_Onset", "FunctionWord_Onset", "Word2_Onset"]
for col in int_cols:
    if col in df_merged.columns:
        df_merged[col] = df_merged[col].astype("Int64")

# === Save merged dataframe ===
save_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Behavioral_With_Correct_Onsets.csv"
df_merged.to_csv(save_path, index=False)

print(f"✅ Merged and saved to: {save_path}")
print("🧾 Preview:")
display(df_merged.head(100))


#2. Feature Engineering

### Feature Preprocessing

###Split the regions for each sentence

In [ ]:

# === Load and parse Sentences_List1.txt ===
def parse_sentence_components(txt_path):
    with open(txt_path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    structured_data = []

    for line in lines:
        line = line.strip()
        if line and line[0].isdigit() and '|' in line:
            try:
                index_part, sentence = line.split('.', 1)
                index = int(index_part.strip())

                # Split the sentence into chunks using |
                chunks = sentence.strip().split('|')

                # Safety check
                if len(chunks) < 4:
                    print(f"⚠️ Skipping line {index}: not enough | segments.")
                    continue

                # Extract components
                preceding_context = chunks[0].strip()
                target_raw = chunks[1].strip()
                following_context = chunks[2].strip()
                extra_context = chunks[3].strip()

                # Target region is structured: Word1 + Function Word + start of Word2
                target_tokens = target_raw.split()
                if len(target_tokens) < 3:
                    print(f"⚠️ Skipping line {index}: target region has < 3 tokens.")
                    continue

                word1 = target_tokens[0]
                function_word = target_tokens[1]
                word2_start = target_tokens[2]

                target_region = f"{word1} {function_word} {word2_start}"

                structured_data.append({
                    "Index": index,
                    "Preceding_Context": preceding_context,
                    "Target_Region": target_region,
                    "Following_Context": following_context,
                    "Extra_Context": extra_context
                })

            except Exception as e:
                print(f"❌ Error parsing line: {line}\n{e}")
                continue

    return pd.DataFrame(structured_data)

# === Path to your sentence file ===
sentence_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/Sentences_List1.txt"

# === Run the parser ===
df_sentences = parse_sentence_components(sentence_path)

# === Save parsed output ===
csv_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Sentences_List1_parsed.csv"
df_sentences.to_csv(csv_path, index=False)

# Optional: save as Pickle (faster for later use in Python)
pkl_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Sentences_List1_parsed.pkl"
df_sentences.to_pickle(pkl_path)

# === Display the first few rows ===
from IPython.display import display
display(df_sentences.head(10))






### Feature 1: Speech Rate of Preceding Context

In [ ]:

# === File paths ===
parsed_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Sentences_List1_parsed.csv"
onset_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/SR_OnsetTimes_Final.csv"

# === Load parsed sentences ===
df_sentences = pd.read_csv(parsed_path)

# === Load onset times and rename columns for consistency ===
df_onsets = pd.read_csv(onset_path)
df_onsets.rename(columns={
    "sentence#": "Index",
    "critical": "Target_Onset_ms"
}, inplace=True)

# === Merge on Index ===
df = pd.merge(df_sentences, df_onsets, on="Index")

# === Convert onset time to seconds ===
df["Preceding_Duration_sec"] = df["Target_Onset_ms"] / 1000.0

# === Count syllables in Preceding_Context ===
dic = pyphen.Pyphen(lang='en')

def count_syllables(text):
    return sum(len(dic.inserted(word).split('-')) for word in text.split())

df["Preceding_Syllables"] = df["Preceding_Context"].apply(count_syllables)

# === Compute Distal Speech Rate ===
df["SpeechRate_Preceding"] = df["Preceding_Syllables"] / df["Preceding_Duration_sec"]

# === Display result ===
display(df[["Index", "Preceding_Context", "Preceding_Syllables", "Preceding_Duration_sec", "SpeechRate_Preceding"]].head(10))

# === Save final DataFrame to CSV ===
output_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Sentences_with_DistalSpeechRate.csv"
df.to_csv(output_path, index=False)
print(f"Data saved to: {output_path}")



###  Acoustic Features Predicting Perception


In [ ]:

# === Load merged dataset (with onset timings and behavioral responses) ===
df_merged = pd.read_csv("/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Behavioral_With_Correct_Onsets.csv")

# === Load and clean distal speech rate data ===
df_rate = pd.read_csv("/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Sentences_with_DistalSpeechRate.csv")
df_rate.columns = df_rate.columns.str.strip()

# Ensure consistent formatting and types for merging
df_rate["Index"] = df_rate["Index"].astype(int)
df_rate["Condition"] = df_rate["NorE"].str.upper().str.strip()

df_merged["Index"] = df_merged["Index"].astype(int)
df_merged["Condition"] = df_merged["Condition"].str.upper().str.strip()

# === Merge in the speech rate info using Index and Condition ===
df_merged = df_merged.drop(columns=[col for col in df_merged.columns if "SpeechRate_Preceding" in col], errors="ignore")

df_merged = df_merged.merge(
    df_rate[["Index", "Condition", "SpeechRate_Preceding"]],
    on=["Index", "Condition"],
    how="left"
)

# === Compute interval between FunctionWord and Word2 ===
df_merged["FuncToWord2Interval"] = (df_merged["Word2_Onset"] - df_merged["FunctionWord_Onset"]) / 1000.0

# === Select relevant features and drop rows with missing values ===
features_df = df_merged[[
    "SpeechRate_Preceding",
    "FuncToWord2Interval",
    "Correct_Sequence (0/1)"
]].dropna()

# Convert label to integer type
features_df["Correct_Sequence (0/1)"] = features_df["Correct_Sequence (0/1)"].astype(int)

# === Preview ===
print("🔍 Feature sample:")
display(features_df.head(40))

# === Save features for modeling ===
features_df.to_csv(
    "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Acoustic_Features.csv",
    index=False
)
print("💾 Saved cleaned feature data.")


#3. Inference:

## Accoustic Features Only

### Logistic Regression

In [ ]:

# === Step 1: Prepare X and y ===
X = features_df[["SpeechRate_Preceding", "FuncToWord2Interval"]]
y = features_df["Correct_Sequence (0/1)"]

# === Step 2: Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# === Step 3: Fit logistic regression ===
model = LogisticRegression()
model.fit(X_train, y_train)

# === Step 4: Predict and evaluate ===
y_pred = model.predict(X_test)

print("📊 Classification Report:\n")
print(classification_report(y_test, y_pred))

# === Step 5: Confusion matrix ===
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
            xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# === Step 6: Check coefficients ===
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})
print("\n🧠 Feature Importance (Logistic Coefficients):")
display(coef_df)


### XGBoost

In [ ]:
# === Step 1: Prepare X and y ===
X = features_df[["SpeechRate_Preceding", "FuncToWord2Interval"]]
y = features_df["Correct_Sequence (0/1)"]

# === Step 2: Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# === Step 3: Fit XGBoost classifier ===
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

# === Step 4: Predict and evaluate ===
y_pred = model.predict(X_test)

print("📊 Classification Report:\n")
print(classification_report(y_test, y_pred))

# === Step 5: Confusion matrix ===
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
            xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# === Step 6: Feature importance ===
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\n🧠 Feature Importance (XGBoost):")
display(importance_df)


## ERP

In [ ]:
# === Configuration ===
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
output_path = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/ERP_Features.csv'


# Time windows (in seconds)
n1_window = (0.09, 0.13)     # ~100 ms
n280_window = (0.25, 0.32)   # ~280 ms

# Optional: specify channels to average over (e.g., central)
# If None, will average over all EEG channels
selected_channels = None  # e.g., ['Cz', 'FCz', 'CPz']

# === Processing loop ===
all_erp_data = []

subject_dirs = sorted([d for d in os.listdir(root_dir) if d.isdigit()], key=lambda x: int(x))

for subj in subject_dirs:
    clean_epo_path = os.path.join(root_dir, subj, f"{subj}_O1_epo_clean.fif")

    if not os.path.exists(clean_epo_path):
        print(f"❌ No cleaned epochs for subject {subj}, skipping.")
        continue

    print(f"📥 Processing subject {subj}...")

    try:
        epochs = mne.read_epochs(clean_epo_path, preload=True)
        times = epochs.times
        data = epochs.get_data()  # shape: (n_trials, n_channels, n_times)

        # Subset channels if specified
        if selected_channels:
            picks = mne.pick_channels(epochs.ch_names, selected_channels)
            data = data[:, picks, :]
        else:
            # Average over all EEG channels (not EOG or misc)
            picks = mne.pick_types(epochs.info, eeg=True)
            data = data[:, picks, :]

        # Get indices for N1 and N280 time windows
        n1_idx = np.where((times >= n1_window[0]) & (times <= n1_window[1]))[0]
        n280_idx = np.where((times >= n280_window[0]) & (times <= n280_window[1]))[0]

        # Extract mean amplitude per trial per window
        n1_amp = data[:, :, n1_idx].mean(axis=(1, 2))
        n280_amp = data[:, :, n280_idx].mean(axis=(1, 2))
        print(f"Size n1_amp: {n1_amp.shape}.\n Size n280_amp: {n280_amp.shape}.")

        # Trial-level dataframe
        n_trials = len(n1_amp)
        df = pd.DataFrame({
            'Participant': [int(subj)] * n_trials,
            'Index': np.arange(n_trials),
            'N1_amplitude': n1_amp,
            'N280_amplitude': n280_amp
        })
        all_erp_data.append(df)

    except Exception as e:
        print(f"⚠️ Error processing subject {subj}: {e}")
        continue

# === Combine and save ===
erp_df = pd.concat(all_erp_data, ignore_index=True)
erp_df.to_csv(output_path, index=False)

print(f"\n✅ Saved ERP feature file: {output_path}")
display(erp_df.head(40))


##ERP + 2 Features

 ### Merge ERP + Acoustic Data

In [ ]:
# === Step 1: Build acoustic feature set ===
# Ensure column names are clean
df_merged.columns = df_merged.columns.str.strip()

# Select relevant columns and drop rows with missing values
features_df = df_merged[[
    "Participant",
    "Index",
    "SpeechRate_Preceding",
    "FuncToWord2Interval",
    "Correct_Sequence (0/1)"
]].dropna()

# Convert target column to integer
features_df["Correct_Sequence (0/1)"] = features_df["Correct_Sequence (0/1)"].astype(int)

print("✅ Acoustic features:")
display(features_df.head())

# === Step 2: Load ERP features ===
erp_path = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/ERP_Features.csv'
erp_df = pd.read_csv(erp_path)
erp_df.columns = erp_df.columns.str.strip()  # Clean column names

print("✅ ERP features:")
display(erp_df.head())

# === Step 3: Merge acoustic + ERP data ===
combined_df = features_df.merge(erp_df, on=["Participant", "Index"], how="inner")
combined_df = combined_df.dropna()

print(f"✅ Combined dataset shape: {combined_df.shape}")
display(combined_df.head(10))


## Classification

### Logistic Regression

In [ ]:
# === Step 1: Define predictors and target ===
X = combined_df[[
    "SpeechRate_Preceding",
    "FuncToWord2Interval",  # ✅ fixed typo here
    "N1_amplitude",
    "N280_amplitude"
]]
y = combined_df["Correct_Sequence (0/1)"]

# === Step 2: Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# === Step 3: Fit logistic regression ===
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# === Step 4: Predict and evaluate ===
y_pred = model.predict(X_test)

print("📊 Classification Report:")
print(classification_report(y_test, y_pred))

# === Step 5: Confusion matrix ===
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
            xticklabels=["Not Perceived", "Perceived"],
            yticklabels=["Not Perceived", "Perceived"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# === Step 6: Feature importances ===
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})
print("🧠 Feature Importances:")
display(coef_df)


Summary:

SpeechRate_Preceding (-0.7578):
A higher preceding speech rate is strongly associated with a lower chance of perceiving the function word correctly. In other words, at normal/fast speeach rates, the chance of hearning the function word is higher. This is the strongest predictor here.

FuncToWord2Interval (-0.0885):
Longer intervals between the function word and the next word slightly reduce the chance of correct perception, but this effect is weaker than speech rate.

N1_amplitude (-0.000009) and N280_amplitude (0.000002):
These ERP amplitudes have very small coefficients, indicating minimal influence on the outcome in this model.        


### XGBoost Classifier

In [ ]:
# === Step 3: Fit XGBoost ===
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train, y_train)

# === Step 4: Predict and evaluate ===
y_pred = xgb_model.predict(X_test)

print("📊 Classification Report:")
print(classification_report(y_test, y_pred))

# === Step 5: Confusion matrix ===
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", xticklabels=["Not Perceived", "Perceived"], yticklabels=["Not Perceived", "Perceived"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()




## Inference ERP + Additional Features

In [ ]:
# Load acoustic summary
acoustic_df = pd.read_csv("/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Extracted_Features/summary_acoustic_features.csv")

# Remove extension and extract numeric part (e.g., from 'SR105.wav' get '105')
acoustic_df["file_id"] = acoustic_df["file"].str.replace(".wav", "", regex=False)
acoustic_df["Index"] = acoustic_df["file_id"].apply(lambda x: int(re.search(r"\d+", x).group()))

# Show result
acoustic_df.head(129)



In [ ]:
# ✅ Step 0: Ensure 'Index' is of the same type
combined_df["Index"] = combined_df["Index"].astype(int)
acoustic_df["Index"] = acoustic_df["Index"].astype(int)

# ✅ Step 1: Remove duplicate/conflicting columns
# If "file" and "file_id" are not needed, drop them to avoid merge conflicts
acoustic_df_clean = acoustic_df.drop(columns=["file", "file_id"], errors="ignore")

# Optional: Also remove any duplicate column names that already exist in combined_df
duplicate_cols = [col for col in acoustic_df_clean.columns if col in combined_df.columns and col != "Index"]
acoustic_df_clean = acoustic_df_clean.drop(columns=duplicate_cols)

# ✅ Step 2: Merge the feature sets
df_all = combined_df.merge(acoustic_df_clean, on="Index", how="left")

# ✅ Step 3: Clean up target column if needed
df_all["Correct_Sequence (0/1)"] = df_all["Correct_Sequence (0/1)"].astype(int)

# ✅ Step 4: Review the final dataset
print("✅ Final shape with all features:", df_all.shape)
display(df_all.head())



### Logistic Regression

In [ ]:
# 🎯 Define target variable
target = "Correct_Sequence (0/1)"

# 🧠 Define feature columns (exclude Participant, Index, and target)
exclude_cols = ["Participant", "Index", target]
feature_cols = [col for col in df_all.columns if col not in exclude_cols]

# 🧹 Drop any rows with missing values
df_model = df_all[feature_cols + [target]].dropna()

# 🪄 Split into train/test
X = df_model[feature_cols]
y = df_model[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 🤖 Train logistic regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 📊 Evaluate
y_pred = model.predict(X_test)
print("📋 Classification Report:")
print(classification_report(y_test, y_pred))

# 📉 Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# 🧠 Print logistic regression coefficients
coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": model.coef_[0]
}).sort_values(by="Coefficient", key=abs, ascending=False)

print("\n🧠 Logistic Regression Coefficients (sorted by absolute value):")
print(coef_df)



Summary:

| Feature                   | Coefficient | Interpretation                                                                                       |
| ------------------------- | ----------- | ---------------------------------------------------------------------------------------------------- |
| **SpeechRate\_Preceding** | -0.9336     | Strong negative effect: higher speech rate → more likely to hear the function word (since 0 = heard) |
| **FuncToWord2Interval**   | -0.2048     | Negative effect: longer interval → more likely to hear function word                                 |
| **MFCC\_12\_Mean**        | -0.1157     | Moderate negative influence — some acoustic cue helping perception                                   |
| **MFCC\_2\_Mean**         | 0.0534      | Positive coefficient means higher value slightly increases chance of **not** hearing function word   |
| **MFCC\_7\_Mean**         | -0.0528     | Small negative effect — helps hearing                                                                |
| **MFCC\_13\_Mean**        | -0.0418     | Small negative effect — helps hearing                                                                |
                                                                                          |
| **N1\_amplitude**         | -0.000021   | Almost no effect here                                                                                |
| **N280\_amplitude**       | -0.000016   | Almost no effect                                                                                     |


### XGBoost Classifier

In [ ]:
# ✅ Set the final merged dataset
df_final = df_all  # Your full dataset with acoustic + ERP + MFCC features

# === Step 1: Define features and target ===
X = df_final.drop(columns=["Correct_Sequence (0/1)", "Participant", "Index"], errors="ignore")
y = df_final["Correct_Sequence (0/1)"]

# === Step 2: Train-test split ===
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

# === Step 3: Scale features ===
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# === Step 4: Train XGBoost classifier ===
from xgboost import XGBClassifier
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_scaled, y_train)

# === Step 5: Evaluate model ===
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
y_pred = xgb_model.predict(X_test_scaled)

print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred))

# === Confusion Matrix ===
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not Perceived", "Perceived"])
disp.plot(cmap="Blues")

# === Step 6: Feature Importance ===
import pandas as pd
import matplotlib.pyplot as plt

importance_scores = xgb_model.feature_importances_
feature_names = X.columns

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_scores
}).sort_values(by="Importance", ascending=False)

print("\n🔝 Top 10 Most Important Features:")
print(importance_df.head(10))

# Plot top 15
plt.figure(figsize=(10, 6))
plt.barh(importance_df["Feature"][:15][::-1], importance_df["Importance"][:15][::-1])
plt.xlabel("Importance Score")
plt.title("Top 15 Most Important Features (XGBoost)")
plt.tight_layout()
plt.show()



Yes — acoustic features and ERP signals together can reliably predict perceptual outcomes.

The model successfully distinguishes perceived from not perceived items with 86% accuracy.

This supports the hypothesis that both neural (ERP) and speech timing/acoustic cues carry meaningful information about perception.

Cross-Validation All features

In [ ]:
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# === Features and target ===
X = df_all.drop(columns=["Correct_Sequence (0/1)", "Participant", "Index"], errors="ignore")
y = df_all["Correct_Sequence (0/1)"]

# === Prepare cross-validation ===
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
scores = []

# Container for all out-of-sample results
all_results = []

# === Cross-validation loop ===
for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train model
    model = xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    # === Metrics ===
    print(f"\n📂 Fold {fold} Classification Report:")
    print(classification_report(y_test, y_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
                xticklabels=["No", "Yes"],
                yticklabels=["No", "Yes"])
    plt.title(f"Confusion Matrix - Fold {fold}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

    # Accuracy for fold
    scores.append(model.score(X_test, y_test))

    # === Store results for error analysis ===
    fold_results = X_test.copy()
    fold_results['y_true'] = y_test.values
    fold_results['y_pred'] = y_pred
    fold_results['correct'] = (fold_results['y_true'] == fold_results['y_pred']).astype(int)
    all_results.append(fold_results)

    fold += 1

# === Combine all folds into one DataFrame for error analysis ===
all_results_df = pd.concat(all_results, axis=0).reset_index(drop=True)

print("\n✅ Average Accuracy Across Folds:", np.mean(scores))
print(f"📊 all_results contains {all_results_df.shape[0]} rows of out-of-sample predictions.")


Summary:

Balanced precision and recall across both perceived and non-perceived trials suggest the model does not favor one class over the other. The model consistently performs well across folds, indicating stable generalization. This shows that acoustic, prosodic, and ERP features together can reliably predict perception of the function word.

### SHAP

In [ ]:

# === Step 7: SHAP Interpretability ===
import shap

# Create SHAP explainer using the scaled training data
explainer = shap.Explainer(xgb_model, X_train_scaled)

# Compute SHAP values for the test set
shap_values = explainer(X_test_scaled)

# === Summary plot: Global feature impact ===
shap.summary_plot(shap_values, X_test_scaled, feature_names=X.columns)


The model’s predictions are primarily driven by the preceding speech rate, which has the strongest influence on function word perception. Additionally, neural components such as the N280 and N1 amplitudes, along with several key acoustic features, contribute meaningfully to the model’s performance. Lower-level spectral features, while still relevant, play a more subtle role. Overall, these results suggest that both behavioral timing factors and specific neural and acoustic characteristics jointly impact the accuracy of function word perception.


##XGBoost Feature Selection

### XGBoost 4 Features

In [ ]:



# Select the top 4 features
top_features = ["ZCR_Mean", "FuncToWord2Interval", "N280_amplitude", "N1_amplitude"]

# Prepare X and y
X = df_all[top_features]
y = df_all["Correct_Sequence (0/1)"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Initialize and train XGBoost
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)

print("📊 Classification Report:\n")
print(classification_report(y_test, y_pred))

# Confusion matrix plot
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
            xticklabels=["Heard", "Did Not Hear"], yticklabels=["Heard", "Did Not Hear"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# Feature importance plot
importance_df = pd.DataFrame({
    "Feature": top_features,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\n🧠 Feature Importance (XGBoost):")
display(importance_df)


Cross-validation 4 Features

In [ ]:


# === Top 4 Features ===
top_features = ["ZCR_Mean", "FuncToWord2Interval", "N280_amplitude", "N1_amplitude"]

# Features and target
X = df_all[top_features]
y = df_all["Correct_Sequence (0/1)"]

# Prepare cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
scores = []

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print(f"\n📂 Fold {fold} Classification Report:")
    print(classification_report(y_test, y_pred))

    # === Confusion Matrix with Cleaner Labels ===
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
                xticklabels=["No", "Yes"],
                yticklabels=["No", "Yes"])
    plt.title(f"Confusion Matrix - Fold {fold}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

    scores.append(model.score(X_test, y_test))
    fold += 1

print("\n✅ Average Accuracy Across Folds (4 features):", np.mean(scores))


### XGBoost 5 Features

In [ ]:


# Original 4 features
base_features = ["ZCR_Mean", "FuncToWord2Interval", "N280_amplitude", "N1_amplitude"]

# Add one more feature
feature_to_add = "SpeechRate_Preceding"
features = base_features + [feature_to_add]

print(f"Training with features: {features}")

# Prepare X and y
X = df_all[features]
y = df_all["Correct_Sequence (0/1)"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train model
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)

print("📊 Classification Report:\n")
print(classification_report(y_test, y_pred))

# Confusion matrix plot
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
            xticklabels=["Heard", "Did Not Hear"], yticklabels=["Heard", "Did Not Hear"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# Feature importance plot
importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\n🧠 Feature Importance (XGBoost):")
display(importance_df)



Cross-validation 5 Features

In [ ]:

# Define the 5 features
features = ["ZCR_Mean", "FuncToWord2Interval", "N280_amplitude", "N1_amplitude", "SpeechRate_Preceding"]

print(f"Running cross-validation for features: {features}")

# Prepare X and y
X = df_all[features]
y = df_all["Correct_Sequence (0/1)"]

# Cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
scores = []

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print(f"\n📂 Fold {fold} Classification Report:")
    print(classification_report(y_test, y_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
                xticklabels=["No", "Yes"],
                yticklabels=["No", "Yes"])
    plt.title(f"Confusion Matrix - Fold {fold}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

    scores.append(model.score(X_test, y_test))
    fold += 1

print("\n✅ Average Accuracy Across Folds:", np.mean(scores))


### XGBoost 6 Features

In [ ]:


# Original 4 features
base_features = ["ZCR_Mean", "FuncToWord2Interval", "N280_amplitude", "N1_amplitude"]

# Add two more features
features_to_add = ["SpeechRate_Preceding", "F0_Mean"]
features = base_features + features_to_add

print(f"Training with features: {features}")

# Prepare X and y
X = df_all[features]
y = df_all["Correct_Sequence (0/1)"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train model
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)

print("📊 Classification Report:\n")
print(classification_report(y_test, y_pred))

# Confusion matrix plot
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
            xticklabels=["Heard", "Did Not Hear"], yticklabels=["Heard", "Did Not Hear"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# Feature importance plot
importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\n🧠 Feature Importance (XGBoost):")
display(importance_df)


Cross-validation 6 Features

In [ ]:
# Original 4 features + 2 more
base_features = ["ZCR_Mean", "FuncToWord2Interval", "N280_amplitude", "N1_amplitude"]
features_to_add = ["SpeechRate_Preceding", "F0_Mean"]
features = base_features + features_to_add

print(f"Running cross-validation for features: {features}")

# Prepare X and y
X = df_all[features]
y = df_all["Correct_Sequence (0/1)"]

# Cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
scores = []

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print(f"\n📂 Fold {fold} Classification Report:")
    print(classification_report(y_test, y_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
                xticklabels=["No", "Yes"],
                yticklabels=["No", "Yes"])
    plt.title(f"Confusion Matrix - Fold {fold}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

    scores.append(model.score(X_test, y_test))
    fold += 1

print("\n✅ Average Accuracy Across Folds:", np.mean(scores))



Model Performance Summary

| Features Used | Accuracy | Precision (Class 0) | Recall (Class 0) | F1-Score (Class 0) | Precision (Class 1) | Recall (Class 1) | F1-Score (Class 1) | Key Feature Importance         |
| ------------- | -------- | ------------------- | ---------------- | ------------------ | ------------------- | ---------------- | ------------------ | ------------------------------ |
| 4 features    | 0.73     | 0.71                | 0.75             | 0.73               | 0.75                | 0.70             | 0.72               | FuncToWord2Interval (0.3226)   |
| 5 features    | 0.82     | 0.79                | 0.86             | 0.82               | 0.85                | 0.78             | 0.81               | SpeechRate\_Preceding (0.4348) |
| 6 features    | 0.81     | 0.77                | 0.88             | 0.82               | 0.86                | 0.75             | 0.80               | SpeechRate\_Preceding (0.3945) |
| All features  | 0.86     | 0.87                | 0.83             | 0.85               | 0.85                | 0.88             | 0.86               | SpeechRate\_Preceding (0.1302) |


## Permutation test XGBoost All Features

In [ ]:
df_final = df_final

# === Step 1: Define features and target ===
X = df_final.drop(columns=["Correct_Sequence (0/1)", "Participant", "Index"], errors="ignore")
y = df_final["Correct_Sequence (0/1)"]

# === Step 2: Train-test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

# === Step 3: Scale features ===
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# === Step 4: Train XGBoost classifier ===
xgb_model = XGBClassifier(eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_scaled, y_train)

# === Step 5: Evaluate real model ===
y_pred = xgb_model.predict(X_test_scaled)
real_accuracy = accuracy_score(y_test, y_pred)
print(f"\n✅ Real model accuracy: {real_accuracy:.3f}")

# === Step 6: Permutation Test ===
n_permutations = 1000
permuted_accuracies = []

for i in range(n_permutations):
    y_permuted = np.random.permutation(y)

    X_train_perm, X_test_perm, y_train_perm, y_test_perm = train_test_split(
        X, y_permuted, test_size=0.3, stratify=y_permuted, random_state=42)

    X_train_perm_scaled = scaler.fit_transform(X_train_perm)
    X_test_perm_scaled = scaler.transform(X_test_perm)

    model_perm = XGBClassifier(eval_metric='logloss', random_state=42)
    model_perm.fit(X_train_perm_scaled, y_train_perm)

    y_pred_perm = model_perm.predict(X_test_perm_scaled)
    acc_perm = accuracy_score(y_test_perm, y_pred_perm)
    permuted_accuracies.append(acc_perm)

# === Step 7: Calculate p-value ===
p_value = np.mean(np.array(permuted_accuracies) >= real_accuracy)
print(f"🔬 Permutation test p-value: {p_value:.5f}")

# === Step 8: Plot results ===
plt.figure(figsize=(8, 5))
plt.hist(permuted_accuracies, bins=30, alpha=0.7, edgecolor='black')
plt.axvline(real_accuracy, color='red', linestyle='dashed', linewidth=2,
            label=f"Real Accuracy = {real_accuracy:.2f}")
plt.title("Permutation Test: Accuracy Distribution (Null Hypothesis)")
plt.xlabel("Accuracy")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()


A permutation test (1,000 iterations) confirmed that the XGBoost classifier significantly outperformed chance (real accuracy = 0.86; null mean ≈ 0.51; p < 0.001).
Therefore, we reject the null hypothesis (H₀) that ERP and acoustic features do not predict perception accuracy.
Instead, we accept the alternative hypothesis (H₁), concluding that ERP and acoustic features meaningfully predict perception accuracy.

# Results/Conlusion


### Models Evaluated
We evaluated two main classification models to predict participants’ perception of function words:  
- **Logistic Regression**  
- **XGBoost Classifier**  

---

### Features Used
The models were initially trained using various subsets of features, including ERP and acoustic predictors.  
Although we performed exploratory feature selection, the **best-performing model** (highest accuracy) was obtained using **all available features**:  

- **Acoustic features:**  
  - `SpeechRate_Preceding` (speech rate in the preceding context)  
  - `FuncToWord2Interval` (interval between the function word and the following word)  
  - Mel-Frequency Cepstral Coefficients (MFCCs)  
  - Zero Crossing Rate (ZCR)  
  - Fundamental Frequency (F0), RMS, and Spectral Centroid  

- **ERP (Neural) features:**  
  - `N1_amplitude`  
  - `N280_amplitude`  

---

### Baseline / Null Model
The **null hypothesis** ($H_0$) stated that these features would **not** predict participants' perception accuracy better than chance (~50% accuracy for balanced classes).  
The **alternative hypothesis** ($H_1$) posited that ERP and acoustic features would significantly predict perception accuracy above chance.  

---

### Model Performance
- **Logistic Regression:** ~75% accuracy on the test set using ERP and acoustic features, showing reasonable predictive power and interpretable linear relationships.  
- **XGBoost Classifier (all features):** ~84–85% accuracy, effectively capturing nonlinear interactions among features.  
- **Cross-validation:** Consistent performance across folds, indicating robustness.  

These results show that while feature selection provided some insights, retaining **all features yielded the strongest predictive performance**, especially for XGBoost.  

---

### Permutation Test
We conducted a **permutation test (1,000 iterations)** on the best-performing XGBoost model using **all ERP and acoustic features**:  
- **Real model accuracy:** 0.86  
- **Null distribution mean:** ≈ 0.51  
- **p-value:** $< 0.001$  

Therefore, we **reject the null hypothesis** ($H_0$) and accept the **alternative hypothesis** ($H_1$), confirming that ERP and acoustic features meaningfully predict perception accuracy.  

---

### Feature Importance and Interpretation
- **SpeechRate_Preceding:** Strongest predictor. Faster preceding speech rate increased the likelihood of correctly perceiving function words.  
- **FuncToWord2Interval:** Negative but smaller contribution—shorter intervals improve perception.  
- **ERP components (N1, N280):** Marginal but meaningful contributions, indicating neurophysiological correlates of auditory perception.  
- **Acoustic features (MFCCs, ZCR, F0, RMS, Spectral Centroid):** Added spectral and temporal speech information.  
- **XGBoost feature importance:** Heavily weighted SpeechRate_Preceding and FuncToWord2Interval, while also integrating ERP and acoustic features.  

---

### Conclusion
This study shows that **the integration of ERP and acoustic features** significantly predicts perception accuracy.  
The permutation test confirms that the best-performing model—XGBoost with **all features**—performs far above chance, leading to a clear **rejection of $H_0$**.  

Key insights:  
- Speech rate plays a dominant role in speech perception.  
- ERP features, while smaller in effect, enhance prediction.  
- Combining all features provides the highest accuracy and best overall performance.  

Future work should explore larger datasets and additional neural features for even better predictive modeling.  


